# 01 - 环境搭建与模型准备

在DGX Spark上完成环境配置、依赖安装和基座模型下载。

**DGX Spark硬件**: GB10 Grace Blackwell | 128GB Unified Memory | 1 PFLOPS FP4

## 1.1 检查硬件环境

In [ ]:
# 检查GPU和内存
import torch
import os

print("="*60)
print("DGX Spark 硬件检查")
print("="*60)
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"GPU数量: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}")
        print(f"  总内存: {props.total_memory / 1024**3:.1f} GB")
    print(f"当前内存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# 检查CPU内存
import psutil
mem = psutil.virtual_memory()
print(f"CPU总内存: {mem.total / 1024**3:.1f} GB")
print(f"可用内存: {mem.available / 1024**3:.1f} GB")
print("="*60)

## 1.2 安装依赖包

In [ ]:
# 安装依赖 (仅需运行一次)
!pip install -q transformers peft bitsandbytes accelerate datasets trl
!pip install -q evaluate scikit-learn matplotlib seaborn tqdm
!pip install -q jieba huggingface-hub

# 验证安装
import transformers
import peft
import bitsandbytes
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print("\n所有依赖安装完成!")

## 1.3 加载配置文件

In [ ]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

from config import (
    BASE_MODEL, MODELS_DIR, DATA_DIR, OUTPUTS_DIR,
    QLORA_CONFIG, DATA_CONFIG, BENCHMARK_CONFIG, HARDWARE_CONFIG
)

print("="*60)
print("配置信息")
print("="*60)
print(f"基座模型: {BASE_MODEL}")
print(f"模型目录: {MODELS_DIR}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUTS_DIR}")
print(f"\nQLoRA配置:")
print(f"  rank: {QLORA_CONFIG['lora']['r']}")
print(f"  alpha: {QLORA_CONFIG['lora']['lora_alpha']}")
print(f"  dropout: {QLORA_CONFIG['lora']['lora_dropout']}")
print(f"\n硬件配置:")
print(f"  设备: {HARDWARE_CONFIG['device']}")
print(f"  GPU内存: {HARDWARE_CONFIG['gpu_memory']} GB")
print("="*60)

## 1.4 加载基座模型

In [ ]:
from utils.training_utils import load_model_and_tokenizer

# 模型路径 (DGX Spark上可能已预下载到本地)
local_model_path = os.path.join(MODELS_DIR, BASE_MODEL.split("/")[-1])

if os.path.exists(local_model_path):
    model_path = local_model_path
    print(f"使用本地模型: {model_path}")
else:
    model_path = BASE_MODEL
    print(f"从HuggingFace下载: {model_path}")

# 加载模型和分词器
model, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=None,
    device_map="auto",
    trust_remote_code=True,
)

print("\n模型加载成功!")

## 1.5 测试模型推理

In [ ]:
# 简单推理测试
test_prompt = "请简要介绍TRIZ方法论的核心思想:"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("模型回复:")
print(response)

# 清理显存
del model
del tokenizer
torch.cuda.empty_cache()
print("\n显存已清理")

---

## 下一步

环境搭建完成！接下来请打开: **02_data_preparation.ipynb** 准备训练数据